# Audio Perturbation Study

How pitch shifts and additive noise affect three signals:
- **ASR quality** (WER, CER) — Whisper small
- **Text embedding similarity** (Jina v4) — embedding of ASR output vs. ground-truth text
- **Audio embedding similarity** (AttentionPool) — augmented audio vs. clean audio

Dataset: ADMED Voice (Polish, natural speech)

| Axis | Values |
|------|--------|
| Noise (white, SNR dB) | clean, 30, 20, 10, 5 |
| Pitch (semitones) | −4, −2, 0, +2, +4 |

In [ ]:
%pip install -q -e '..'

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

from evaluator.datasets.core import load_admed_voice_corpus, AdmedQueryDataset
from evaluator.config.audio_augmentation import AudioAugmentationConfig
from evaluator.pipeline.audio.augmentation import AudioAugmenter
from evaluator.models.asr.whisper import WhisperModel
from evaluator.models.t2e.jina import JinaV4Model
from evaluator.models.a2e.attention_pool import AttentionPoolAudioModel
from evaluator.metrics import word_error_rate, character_error_rate, term_weighted_wer

# Domain-aware error weighting: mishearing a medical term costs more than a stop word.
# term_weighted_wer weights each reference word by this table (default 1.0 otherwise),
# so we can watch *clinical* terms degrade under perturbation, not just overall WER.
MEDICAL_TERM_WEIGHTS = {
    "hypertension": 3.0, "diabetes": 3.0, "tachycardia": 3.0, "dyspnea": 3.0,
    "myocardial": 3.0, "infarction": 3.0, "ischemia": 3.0, "edema": 3.0,
    "pneumonia": 3.0, "sepsis": 3.0, "embolism": 3.0, "arrhythmia": 3.0,
}

In [ ]:
N_SAMPLES       = 30   # samples from ADMED test split
HEATMAP_SAMPLES = 15   # smaller subset for 2-D grid (faster)
LANGUAGE        = "pl"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")

## 1. Load ADMED data

> **Note:** `AdmedQueryDataset` is a low-level/deprecated loader, kept here for a
> per-sample study. For normal evaluation prefer the dataset-descriptor registry —
> set `data.dataset_type` in an `EvaluationConfig` and let `run_evaluation` load it.

In [ ]:
_, test_df = load_admed_voice_corpus(test_size=0.3)
dataset = AdmedQueryDataset(test_df)

samples = []
for i in range(min(N_SAMPLES, len(dataset))):
    try:
        samples.append(dataset[i])
    except Exception as e:
        print(f"  skip {i}: {e}")

print(f"loaded {len(samples)} samples")
s0 = samples[0]
print(f"  audio shape : {s0['audio_array'].shape}")
print(f"  sampling_rate: {s0['sampling_rate']}")
print(f"  transcription: {s0['transcription'][:80]}")

## 2. Load models

In [ ]:
asr_model = WhisperModel(model_name="openai/whisper-small")
asr_model.to(device)
print("Whisper small loaded")

jina_model = JinaV4Model()
jina_model.to(device)
print("Jina v4 loaded")

# AttentionPool uses Whisper encoder as backbone; model_path=None → untrained projection
audio_emb_model = AttentionPoolAudioModel(audio_encoder_name="openai/whisper-small")
audio_emb_model.to(device)
print("AttentionPool loaded")

## 3. Pre-compute clean baselines

All similarity metrics compare against these references:
- **Jina sim**: `Jina(ground_truth_text)` vs `Jina(ASR(augmented_audio))`
- **Audio sim**: `AttentionPool(clean_audio)` vs `AttentionPool(augmented_audio)`

In [ ]:
clean_tensors = [torch.from_numpy(s["audio_array"]).float() for s in samples]
clean_srs     = [s["sampling_rate"] for s in samples]
clean_texts   = [s["transcription"] for s in samples]

print("Transcribing clean audio (ASR baseline)...")
clean_hyps = asr_model.transcribe(clean_tensors, clean_srs, language=LANGUAGE)
baseline_wer = np.mean([word_error_rate(r, h) for r, h in zip(clean_texts, clean_hyps)])
print(f"  baseline WER (clean audio): {baseline_wer:.3f}")

print("Encoding clean audio embeddings (AttentionPool)...")
clean_audio_embs = audio_emb_model.encode_audio(clean_tensors, clean_srs, show_progress=True)
print(f"  shape: {clean_audio_embs.shape}")

print("Encoding ground-truth text embeddings (Jina)...")
ref_text_embs = jina_model.encode(clean_texts, show_progress=True)
print(f"  shape: {ref_text_embs.shape}")

## 4. Augmentation helpers

In [ ]:
def make_augmenter(snr_db=None, pitch_semitones=None):
    """Fixed-point augmenter. snr_db=None → no noise. pitch_semitones=None/0 → no shift."""
    use_noise = snr_db is not None
    use_pitch = pitch_semitones is not None and pitch_semitones != 0
    return AudioAugmenter(AudioAugmentationConfig(
        enabled=True,
        add_noise=use_noise,
        noise_type="white",
        snr_db=float(snr_db) if use_noise else 20.0,
        pitch_shift=use_pitch,
        # (n, n) forces a fixed shift instead of random draw
        pitch_semitones_range=(pitch_semitones, pitch_semitones) if use_pitch else (-2, 2),
    ))


def cos_sim_rows(A: np.ndarray, B: np.ndarray) -> np.ndarray:
    """Row-wise cosine similarity. A, B: (N, D) → (N,)"""
    norm_A = np.linalg.norm(A, axis=1, keepdims=True) + 1e-10
    norm_B = np.linalg.norm(B, axis=1, keepdims=True) + 1e-10
    return np.sum((A / norm_A) * (B / norm_B), axis=1)


def augment_tensors(subset, snr_db=None, pitch_semitones=None):
    """Return (tensors, srs) for a sample subset under given augmentation."""
    is_clean = snr_db is None and (pitch_semitones is None or pitch_semitones == 0)
    if is_clean:
        return (
            [torch.from_numpy(s["audio_array"]).float() for s in subset],
            [s["sampling_rate"] for s in subset],
        )
    aug = make_augmenter(snr_db=snr_db, pitch_semitones=pitch_semitones)
    tensors, srs = [], []
    for s in subset:
        augmented = aug.augment(s["audio_array"], s["sampling_rate"])
        tensors.append(torch.from_numpy(augmented).float())
        srs.append(s["sampling_rate"])
    return tensors, srs

In [ ]:
def run_sweep(conditions, subset_size=None):
    """
    conditions: list of (label, snr_db, pitch_semitones)
    Returns DataFrame with per-condition mean +/- std metrics.
    """
    n      = subset_size or len(samples)
    subset = samples[:n]
    ref_texts_sub    = [s["transcription"] for s in subset]
    ref_embs_sub     = ref_text_embs[:n]     # Jina(ground_truth)
    clean_audio_sub  = clean_audio_embs[:n]  # AttentionPool(clean)

    rows = []
    for label, snr_db, pitch in tqdm(conditions, desc="sweep"):
        tensors, srs = augment_tensors(subset, snr_db=snr_db, pitch_semitones=pitch)

        hyps  = asr_model.transcribe(tensors, srs, language=LANGUAGE)
        wers  = [word_error_rate(r, h)      for r, h in zip(ref_texts_sub, hyps)]
        cers  = [character_error_rate(r, h) for r, h in zip(ref_texts_sub, hyps)]
        # domain-weighted: how badly do *medical terms* degrade vs overall WER?
        twwers = [term_weighted_wer(r, h, MEDICAL_TERM_WEIGHTS)
                  for r, h in zip(ref_texts_sub, hyps)]

        hyp_embs   = jina_model.encode(hyps)
        jina_sims  = cos_sim_rows(ref_embs_sub, hyp_embs)

        audio_embs = audio_emb_model.encode_audio(tensors, srs)
        audio_sims = cos_sim_rows(clean_audio_sub, audio_embs)

        rows.append(dict(
            label      = label,
            wer_mean   = np.mean(wers),       wer_std    = np.std(wers),
            twwer_mean = np.mean(twwers),     twwer_std  = np.std(twwers),
            cer_mean   = np.mean(cers),       cer_std    = np.std(cers),
            jina_mean  = np.mean(jina_sims),  jina_std   = np.std(jina_sims),
            audio_mean = np.mean(audio_sims), audio_std  = np.std(audio_sims),
        ))

    return pd.DataFrame(rows)

## 5. Noise sweep

In [ ]:
noise_conditions = [
    ("clean",    None, None),
    ("SNR=30",   30.0, None),
    ("SNR=20",   20.0, None),
    ("SNR=10",   10.0, None),
    ("SNR=5",     5.0, None),
]

noise_df = run_sweep(noise_conditions)
noise_df

## 6. Pitch sweep

In [ ]:
pitch_conditions = [
    ("-4 st", None, -4),
    ("-2 st", None, -2),
    (" 0 st", None,  0),
    ("+2 st", None,  2),
    ("+4 st", None,  4),
]

pitch_df = run_sweep(pitch_conditions)
pitch_df

## 7. Plots — noise sweep

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("Effect of white noise on ASR and embeddings", fontsize=13)

x = noise_df["label"]

axes[0].bar(x, noise_df["wer_mean"], yerr=noise_df["wer_std"], capsize=4, color="steelblue")
axes[0].set_title("WER (Whisper small)")
axes[0].set_ylabel("WER")
axes[0].set_ylim(0, None)

axes[1].bar(x, noise_df["jina_mean"], yerr=noise_df["jina_std"], capsize=4, color="darkorange")
axes[1].set_title("Jina text emb. similarity\n(ASR output vs. ground truth)")
axes[1].set_ylabel("Cosine similarity")
axes[1].set_ylim(0, 1)

axes[2].bar(x, noise_df["audio_mean"], yerr=noise_df["audio_std"], capsize=4, color="seagreen")
axes[2].set_title("AttentionPool audio emb. similarity\n(augmented vs. clean)")
axes[2].set_ylabel("Cosine similarity")
axes[2].set_ylim(0, 1)

for ax in axes:
    ax.tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.savefig("noise_sweep.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Plots — pitch sweep

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("Effect of pitch shift on ASR and embeddings", fontsize=13)

x = pitch_df["label"]

axes[0].bar(x, pitch_df["wer_mean"], yerr=pitch_df["wer_std"], capsize=4, color="steelblue")
axes[0].set_title("WER (Whisper small)")
axes[0].set_ylabel("WER")
axes[0].set_ylim(0, None)

axes[1].bar(x, pitch_df["jina_mean"], yerr=pitch_df["jina_std"], capsize=4, color="darkorange")
axes[1].set_title("Jina text emb. similarity\n(ASR output vs. ground truth)")
axes[1].set_ylabel("Cosine similarity")
axes[1].set_ylim(0, 1)

axes[2].bar(x, pitch_df["audio_mean"], yerr=pitch_df["audio_std"], capsize=4, color="seagreen")
axes[2].set_title("AttentionPool audio emb. similarity\n(augmented vs. clean)")
axes[2].set_ylabel("Cosine similarity")
axes[2].set_ylim(0, 1)

for ax in axes:
    ax.tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.savefig("pitch_sweep.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. Combined heatmap: pitch × noise

Uses `HEATMAP_SAMPLES` subset for speed. Each cell = mean over samples.

In [ ]:
SNR_LEVELS   = [None, 30, 20, 10, 5]       # None = no noise
PITCH_LEVELS = [-4, -2, 0, 2, 4]           # semitones

snr_labels   = ["clean", "30 dB", "20 dB", "10 dB", "5 dB"]
pitch_labels = ["-4 st", "-2 st", "0 st", "+2 st", "+4 st"]

hmap_subset = samples[:HEATMAP_SAMPLES]
ref_texts_h = [s["transcription"] for s in hmap_subset]
ref_embs_h  = ref_text_embs[:HEATMAP_SAMPLES]
clean_embs_h = clean_audio_embs[:HEATMAP_SAMPLES]

wer_grid   = np.zeros((len(PITCH_LEVELS), len(SNR_LEVELS)))
jina_grid  = np.zeros_like(wer_grid)
audio_grid = np.zeros_like(wer_grid)

total = len(PITCH_LEVELS) * len(SNR_LEVELS)
with tqdm(total=total, desc="heatmap grid") as pbar:
    for pi, pitch in enumerate(PITCH_LEVELS):
        for si, snr in enumerate(SNR_LEVELS):
            tensors, srs = augment_tensors(hmap_subset, snr_db=snr, pitch_semitones=pitch)

            hyps  = asr_model.transcribe(tensors, srs, language=LANGUAGE)
            wers  = [word_error_rate(r, h) for r, h in zip(ref_texts_h, hyps)]

            hyp_embs   = jina_model.encode(hyps)
            jina_sims  = cos_sim_rows(ref_embs_h, hyp_embs)

            audio_embs = audio_emb_model.encode_audio(tensors, srs)
            audio_sims = cos_sim_rows(clean_embs_h, audio_embs)

            wer_grid[pi, si]   = np.mean(wers)
            jina_grid[pi, si]  = np.mean(jina_sims)
            audio_grid[pi, si] = np.mean(audio_sims)

            pbar.update(1)

print("Grid done.")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Pitch × Noise interaction", fontsize=14)

heatmap_data = [
    (wer_grid,   "WER",                              "YlOrRd",  False),
    (jina_grid,  "Jina cosine sim (text emb)",       "YlGn",    False),
    (audio_grid, "AttentionPool cosine sim (audio)", "YlGn",    False),
]

for ax, (grid, title, cmap, rev) in zip(axes, heatmap_data):
    cmap_name = cmap + ("_r" if rev else "")
    sns.heatmap(
        pd.DataFrame(grid, index=pitch_labels, columns=snr_labels),
        ax=ax, annot=True, fmt=".2f", cmap=cmap_name,
        linewidths=0.5, cbar_kws={"shrink": 0.8},
    )
    ax.set_title(title)
    ax.set_xlabel("Noise level")
    ax.set_ylabel("Pitch shift")

plt.tight_layout()
plt.savefig("perturbation_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()